<a href="https://colab.research.google.com/github/Risqullah/BIGDATA_5/blob/main/Web_Scraping_BD5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping Berita Agrikultur: FAO, IITA, dan WorldFish
**Sumber**
1. FAO Newsroom — https://www.fao.org/newsroom/archive/
2. IITA News Items — https://www.iita.org/news-item/
3. WorldFish News — https://worldfishcenter.org/news

In [23]:
!pip -q install requests beautifulsoup4 pandas python-dateutil lxml

In [24]:
import re
import time
import hashlib
import warnings

from collections import defaultdict
from datetime import datetime, timezone
from urllib.parse import (
    urljoin,
    urlparse,
    urlencode,
    parse_qs
)
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd

from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", category=FutureWarning)

USER_AGENT = (
    "Mozilla/5.0 "
    "(compatible; StudentAgricultureResearchBot/1.0; educational metadata collection)"
)

REQUEST_DELAY_SECONDS = 2.0
REQUEST_TIMEOUT = 30
MAX_RETRIES_SERVER_ERROR = 1

TARGET_MIN_ROWS = 1000

# Batas halaman pengumpulan awal agar ketiga sumber lebih seimbang.
# Batas ini bukan jumlah artikel yang dijamin.
MAX_PAGES = {
    "FAO": 50,
    "IITA": 119,
    "WorldFish": 50
}

SOURCES = {
    "FAO": {
        "base_url": "https://www.fao.org/newsroom/archive/",
        "robots_url": "https://www.fao.org/robots.txt",
        "kind": "fao",
        "terms_url": "https://www.fao.org/contact-us/terms/en/",
    },

    "IITA": {
        "base_url": "https://iita.org/news-item/",
        "robots_url": "https://iita.org/robots.txt",
        "kind": "iita",
        "terms_url": "https://iita.org/",
    },

    "WorldFish": {
        "base_url": "https://www.worldfishcenter.org/news",
        "robots_url": "https://www.worldfishcenter.org/robots.txt",
        "kind": "worldfish",
        "terms_url": "https://www.worldfishcenter.org/terms-use",
    },
}

session = requests.Session()

session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
})

robots_cache = {}
last_request_time = defaultdict(float)

request_log = []
source_preflight = {}

print("Konfigurasi siap:", ", ".join(SOURCES))
print("Target minimum artikel relevan:", TARGET_MIN_ROWS)
print("Batas halaman:", MAX_PAGES)

Konfigurasi siap: FAO, IITA, WorldFish
Target minimum artikel relevan: 1000
Batas halaman: {'FAO': 50, 'IITA': 119, 'WorldFish': 50}


In [25]:
class RobotsUnverifiedError(RuntimeError):
    pass


class RobotsDeniedError(RuntimeError):
    pass


class SourceBlockedError(RuntimeError):
    pass


def normalized_host(url):
    host = urlparse(url).netloc.lower()

    if host.startswith("www."):
        host = host[4:]

    return host


def get_robots_parser(source_name):
    """
    Mengambil dan membaca robots.txt.
    Jika robots.txt tidak dapat diverifikasi, scraping dihentikan.
    """

    if source_name in robots_cache:
        return robots_cache[source_name]

    robots_url = SOURCES[source_name]["robots_url"]

    try:
        response = session.get(
            robots_url,
            timeout=REQUEST_TIMEOUT
        )
    except requests.RequestException as exc:
        raise RobotsUnverifiedError(
            f"robots.txt tidak dapat diakses untuk {source_name}: {exc}"
        )

    if response.status_code != 200 or not response.text.strip():
        raise RobotsUnverifiedError(
            f"robots.txt {source_name} tidak terverifikasi "
            f"(HTTP {response.status_code})."
        )

    if normalized_host(response.url) != normalized_host(robots_url):
        raise RobotsUnverifiedError(
            f"robots.txt diarahkan ke host berbeda: {response.url}"
        )

    parser = RobotFileParser()
    parser.set_url(robots_url)
    parser.parse(response.text.splitlines())

    robots_cache[source_name] = parser

    print(f"robots.txt {source_name}: berhasil dibaca")

    return parser


def robots_allowed(source_name, url):
    parser = get_robots_parser(source_name)

    return parser.can_fetch(USER_AGENT, url)


def polite_get(source_name, url):
    """
    Mengambil halaman HTML hanya jika diizinkan robots.txt.
    Tidak mencoba melewati blokir atau rate limit.
    """

    if not robots_allowed(source_name, url):
        raise RobotsDeniedError(
            f"robots.txt tidak mengizinkan URL: {url}"
        )

    host = urlparse(url).netloc.lower()

    parser = robots_cache[source_name]

    crawl_delay = (
        parser.crawl_delay(USER_AGENT)
        or parser.crawl_delay("*")
        or 0
    )

    delay = max(
        REQUEST_DELAY_SECONDS,
        float(crawl_delay)
    )

    wait_for = delay - (
        time.time() - last_request_time[host]
    )

    if wait_for > 0:
        time.sleep(wait_for)

    attempts = 0

    while True:
        try:
            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT
            )

            last_request_time[host] = time.time()

            request_log.append({
                "source_website": source_name,
                "url": url,
                "final_url": response.url,
                "http_status": response.status_code,
                "requested_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            })

            if response.status_code in (403, 429):
                raise SourceBlockedError(
                    f"HTTP {response.status_code} dari {url}. "
                    "Proses sumber dihentikan."
                )

            if response.status_code == 404:
                raise SourceBlockedError(
                    f"HTTP 404: {url}"
                )

            if response.status_code >= 500:
                if attempts < MAX_RETRIES_SERVER_ERROR:
                    attempts += 1
                    time.sleep(4 * attempts)
                    continue

            response.raise_for_status()

            content_type = response.headers.get(
                "Content-Type", ""
            ).lower()

            if "html" not in content_type and "text/" not in content_type:
                raise SourceBlockedError(
                    f"Respons bukan HTML: {content_type}"
                )

            # Pastikan redirect tetap berada di domain yang sama.
            if normalized_host(response.url) != normalized_host(url):
                raise SourceBlockedError(
                    f"Redirect menuju domain berbeda: {response.url}"
                )

            requested_path = urlparse(url).path.rstrip("/")
            final_path = urlparse(response.url).path.rstrip("/")

            # Pada IITA, permintaan halaman pagination harus
            # tetap berada pada halaman pagination yang diminta.
            if source_name == "IITA":
                if requested_path != final_path:
                    raise SourceBlockedError(
                        f"URL IITA berubah setelah redirect: "
                        f"{url} -> {response.url}"
                    )

            # Pada WorldFish, nomor halaman harus tetap sama
            # setelah redirect.
            if source_name == "WorldFish":
                requested_page = parse_qs(
                    urlparse(url).query
                ).get("page", [None])[0]

                final_page = parse_qs(
                    urlparse(response.url).query
                ).get("page", [None])[0]

                if (
                    requested_page is not None
                    and requested_page != final_page
                ):
                    raise SourceBlockedError(
                        f"Nomor halaman WorldFish berubah: "
                        f"{url} -> {response.url}"
                    )

            return response.text

        except SourceBlockedError:
            raise

        except requests.RequestException as exc:
            if attempts < MAX_RETRIES_SERVER_ERROR:
                attempts += 1
                time.sleep(4 * attempts)
                continue

            raise SourceBlockedError(
                f"Gagal mengambil halaman {url}: {exc}"
            )


def build_page_url(source_name, page_number):
    """
    Membuat URL pagination berdasarkan struktur masing-masing website.
    Nomor halaman notebook dimulai dari 1.
    """

    base = SOURCES[source_name]["base_url"]

    if source_name == "FAO":
        if page_number <= 1:
            return base

        return (
            f"https://www.fao.org/newsroom/archive/"
            f"{page_number}/en"
        )

    if source_name == "IITA":
        if page_number <= 1:
            return base

        return urljoin(
            base,
            f"page/{page_number}/"
        )

    if source_name == "WorldFish":
        if page_number <= 1:
            return base

        # Pagination WorldFish menggunakan indeks mulai dari 0.
        return (
            base
            + "?"
            + urlencode({"page": page_number - 1})
        )

    raise ValueError(
        f"Sumber tidak dikenal: {source_name}"
    )


print("Fungsi akses, robots, dan pagination siap.")

Fungsi akses, robots, dan pagination siap.


In [26]:
preflight_rows = []

for source_name in SOURCES:

    first_url = build_page_url(source_name, 1)
    second_url = build_page_url(source_name, 2)

    result = {
        "source_website": source_name,
        "robots_verified": False,
        "page_1_allowed": False,
        "page_2_allowed": False,
        "page_1_http": None,
        "page_2_http": None,
        "page_1_final_url": "",
        "page_2_final_url": "",
        "status": "NOT READY",
        "note": "",
    }

    try:
        get_robots_parser(source_name)

        result["robots_verified"] = True

        result["page_1_allowed"] = robots_allowed(
            source_name,
            first_url
        )

        result["page_2_allowed"] = robots_allowed(
            source_name,
            second_url
        )

        if not result["page_1_allowed"]:
            result["note"] = "Halaman pertama tidak diizinkan robots.txt."

        elif not result["page_2_allowed"]:
            result["note"] = "Halaman kedua tidak diizinkan robots.txt."

        else:
            html_1 = polite_get(
                source_name,
                first_url
            )

            log_1 = request_log[-1]

            html_2 = polite_get(
                source_name,
                second_url
            )

            log_2 = request_log[-1]

            result["page_1_http"] = log_1["http_status"]
            result["page_2_http"] = log_2["http_status"]

            result["page_1_final_url"] = log_1["final_url"]
            result["page_2_final_url"] = log_2["final_url"]

            if log_1["final_url"] == log_2["final_url"]:
                result["note"] = (
                    "Halaman 1 dan 2 diarahkan ke URL yang sama. "
                    "Pagination perlu diperiksa."
                )

            elif len(html_1) > 1000 and len(html_2) > 1000:
                result["status"] = "READY"
                result["note"] = (
                    "Kedua halaman berhasil diakses sebagai HTML "
                    "dan memiliki URL akhir berbeda."
                )

            else:
                result["note"] = "HTML terlalu pendek."

    except Exception as exc:
        result["note"] = (
            f"{type(exc).__name__}: {exc}"
        )

    source_preflight[source_name] = result
    preflight_rows.append(result)


preflight_df = pd.DataFrame(preflight_rows)

display(preflight_df)

ready_sources = [
    name
    for name, info in source_preflight.items()
    if info["status"] == "READY"
]

print("\nSumber yang lolos pemeriksaan:", ready_sources)
print("Sumber NOT READY tidak akan di-scrape.")

robots.txt FAO: berhasil dibaca
robots.txt IITA: berhasil dibaca
robots.txt WorldFish: berhasil dibaca


,source_website,robots_verified,page_1_allowed,page_2_allowed,page_1_http,page_2_http,page_1_final_url,page_2_final_url,status,note
0,FAO,True,True,True,200,200,https://www.fao.org/newsroom/archive/,https://www.fao.org/newsroom/archive/2/en,READY,Kedua halaman berhasil diakses sebagai HTML dan memiliki URL akhir berbeda.
1,IITA,True,True,True,200,200,https://iita.org/news-item/,https://iita.org/news-item/page/2/,READY,Kedua halaman berhasil diakses sebagai HTML dan memiliki URL akhir berbeda.
2,WorldFish,True,True,True,200,200,https://www.worldfishcenter.org/news,https://www.worldfishcenter.org/news?page=1,READY,Kedua halaman berhasil diakses sebagai HTML dan memiliki URL akhir berbeda.



Sumber yang lolos pemeriksaan: ['FAO', 'IITA', 'WorldFish']
Sumber NOT READY tidak akan di-scrape.


In [27]:
MONTH_PATTERN = (
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
    r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|"
    r"Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
)

DATE_PATTERNS = [
    r"\b\d{1,2}/\d{1,2}/\d{4}\b",
    rf"\b\d{{1,2}}\s+{MONTH_PATTERN}\s+\d{{4}}\b",
    rf"\b{MONTH_PATTERN}\s+\d{{1,2}},?\s+\d{{4}}\b",
    r"\b\d{4}-\d{2}-\d{2}\b",
]


def normalize_space(value):
    return re.sub(
        r"\s+",
        " ",
        str(value or "")
    ).strip()


def same_source_domain(url, source_name):
    return (
        normalized_host(url)
        ==
        normalized_host(SOURCES[source_name]["base_url"])
    )


def is_article_link(source_name, href, anchor_text=""):
    parsed = urlparse(href)
    path = parsed.path.rstrip("/")

    if not same_source_domain(href, source_name):
        return False

    if source_name == "FAO":
        return "/newsroom/detail/" in path

    if source_name == "IITA":
        # Hanya URL artikel, bukan URL pagination.
        return bool(
            re.fullmatch(
                r"/news-item/[^/]+",
                path
            )
        )

    if source_name == "WorldFish":
        # WorldFish menggunakan URL blog dan node ID.
        article_patterns = [
            r"/blog/[^/]+",
            r"/node/\d+",
            r"/updates/[^/]+",
            r"/impact-stories?/[^/]+",
            r"/events/[^/]+",
            r"/in-the-news/[^/]+",
            r"/press-releases?/[^/]+",
        ]

        return any(
            re.fullmatch(pattern, path)
            for pattern in article_patterns
        )

    return False


def find_date_in_text(text):
    text = normalize_space(text)

    for pattern in DATE_PATTERNS:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            raw_date = match.group(0)

            try:
                dayfirst = bool(
                    re.match(
                        r"^\d{1,2}/\d{1,2}/\d{4}$",
                        raw_date
                    )
                )

                parsed = date_parser.parse(
                    raw_date,
                    dayfirst=dayfirst
                )

                return (
                    parsed.date().isoformat(),
                    raw_date
                )

            except (ValueError, OverflowError):
                pass

    return None, None


def find_card_container(anchor):
    """
    Mencari wadah kartu artikel terdekat.
    Mengutamakan elemen yang mempunyai tanggal publikasi.
    """

    fallback = (
        anchor.parent
        if anchor.parent
        else anchor
    )

    for parent in anchor.parents:

        if getattr(parent, "name", None) in ("body", "html"):
            break

        text = normalize_space(
            parent.get_text(" ", strip=True)
        )

        if not text or len(text) > 2500:
            continue

        if parent.name == "article":
            return parent

        parsed_date, _ = find_date_in_text(text)

        has_heading = (
            parent.find(["h2", "h3", "h4"]) is not None
        )

        if parsed_date and has_heading:
            return parent

        if parsed_date and len(text) < 1500:
            return parent

        if 30 < len(text) < 1400:
            fallback = parent

    return fallback


def extract_card_title(anchor, card):
    """
    Mengambil judul dari teks tautan.
    Jika kosong, gunakan heading kartu atau alt gambar.
    """

    title = normalize_space(
        anchor.get_text(" ", strip=True)
    )

    if len(title) >= 8:
        return title

    heading = anchor.find_parent(
        ["h1", "h2", "h3", "h4"]
    )

    if heading is None and card is not None:
        heading = card.find(
            ["h1", "h2", "h3", "h4"]
        )

    if heading is not None:
        title = normalize_space(
            heading.get_text(" ", strip=True)
        )

    if len(title) >= 8:
        return title

    image_tag = anchor.find("img")

    if image_tag:
        title = normalize_space(
            image_tag.get("alt", "")
        )

    if len(title) >= 8:
        return title

    return normalize_space(
        anchor.get("title", "")
    )


def extract_author(card_text):
    text = normalize_space(card_text)

    match = re.search(
        r"Posted by\s+(.+?)"
        r"(?=\s+\d+\s+minutes?\s+read\b|$)",
        text,
        flags=re.IGNORECASE
    )

    return (
        normalize_space(match.group(1))
        if match
        else ""
    )


def make_summary(card_text, title, raw_date):
    text = normalize_space(card_text)

    if title:
        text = re.sub(
            re.escape(title),
            " ",
            text,
            count=1,
            flags=re.IGNORECASE
        )

    if raw_date:
        text = re.sub(
            re.escape(raw_date),
            " ",
            text,
            count=1,
            flags=re.IGNORECASE
        )

    text = re.sub(
        r"Posted by\s+.*?"
        r"(?=\s+\d+\s+minutes?\s+read\b|$)",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b\d+\s+minutes?\s+read\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bRead more\b|\bImage\b\s*:?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = normalize_space(text).strip(
        " -–|:;,.·"
    )

    return text[:700]


def parse_listing_page(html, source_name, archive_page_url):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    parsed_rows = []
    seen_urls = set()

    for anchor in soup.find_all("a", href=True):

        href = anchor.get("href", "").strip()

        if not href:
            continue

        absolute_url = urljoin(
            archive_page_url,
            href
        )

        parsed_url = urlparse(absolute_url)

        # Hilangkan fragment URL.
        absolute_url = parsed_url._replace(
            fragment=""
        ).geturl()

        if not is_article_link(
            source_name,
            absolute_url
        ):
            continue

        card = find_card_container(anchor)

        title = extract_card_title(
            anchor,
            card
        )

        if len(title) < 8:
            continue

        if title.lower() in {
            "read more",
            "more",
            "next",
            "previous"
        }:
            continue

        if absolute_url in seen_urls:
            continue

        seen_urls.add(absolute_url)

        card_text = normalize_space(
            card.get_text(" ", strip=True)
            if card
            else title
        )

        published_date, raw_date = find_date_in_text(
            card_text
        )

        summary = make_summary(
            card_text,
            title,
            raw_date
        )

        author = extract_author(card_text)

        parsed_rows.append({
            "source_website": source_name,
            "title": title,
            "published_date": published_date,
            "summary": summary,
            "author": author,
            "source_category": "",
            "article_url": absolute_url,
            "archive_page_url": archive_page_url,
        })

    return parsed_rows, soup


print("Parser baru siap.")

Parser baru siap.


In [28]:
parser_test_rows = []

parser_test_url_sets = {}

for source_name in ready_sources:

    for page_number in (1, 2):

        test_url = build_page_url(
            source_name,
            page_number
        )

        try:
            html = polite_get(
                source_name,
                test_url
            )

            final_url = request_log[-1]["final_url"]

            rows, soup = parse_listing_page(
                html,
                source_name,
                final_url
            )

            article_urls = {
                row["article_url"]
                for row in rows
            }

            parser_test_url_sets[
                (source_name, page_number)
            ] = article_urls

            parser_test_rows.append({
                "source_website": source_name,
                "page": page_number,
                "article_cards_found": len(rows),
                "sample_title": (
                    rows[0]["title"]
                    if rows else ""
                ),
                "sample_date": (
                    rows[0]["published_date"]
                    if rows else ""
                ),
                "sample_url": (
                    rows[0]["article_url"]
                    if rows else ""
                ),
                "final_url": final_url,
            })

        except Exception as exc:

            parser_test_url_sets[
                (source_name, page_number)
            ] = set()

            parser_test_rows.append({
                "source_website": source_name,
                "page": page_number,
                "article_cards_found": 0,
                "sample_title": "",
                "sample_date": "",
                "sample_url": "",
                "final_url": "",
                "error": str(exc),
            })


parser_test_df = pd.DataFrame(
    parser_test_rows
)

display(parser_test_df)

print("\nUji parser selesai.")

,source_website,page,article_cards_found,sample_title,sample_date,sample_url,final_url
0,FAO,1,20,FAO urges job creation and coordinated action to tackle hunger and poverty,2026-11-19,https://www.fao.org/newsroom/detail/fao-urges-job-creation-and-coordinated-action-to-tackle-hunger-and-poverty/en,https://www.fao.org/newsroom/archive/
1,FAO,2,20,World Food Day 2026: Make innovation inclusive and impactful for resilient agrifood systems,2026-10-16,https://www.fao.org/newsroom/detail/world-food-day-2026--make-innovation-inclusive-and-impactful-for-resilient-agrif...,https://www.fao.org/newsroom/archive/2/en
2,IITA,1,11,Early career researchers urged to rethink agricultural productivity as IARSAF holds 29th annual symposium,2026-11-20,https://iita.org/news-item/early-career-researchers-urged-to-rethink-agricultural-productivity-as-iarsaf-holds-29th-...,https://iita.org/news-item/
3,IITA,2,11,Informal markets are shaping safer tomato farming practices in Nigeria,2026-11-13,https://iita.org/news-item/informal-markets-are-shaping-safer-tomato-farming-practices-in-nigeria/,https://iita.org/news-item/page/2/
4,WorldFish,1,49,My Worldfish,None,https://www.worldfishcenter.org/node/8055,https://www.worldfishcenter.org/news
5,WorldFish,2,52,My Worldfish,None,https://www.worldfishcenter.org/node/8055,https://www.worldfishcenter.org/news?page=1



Uji parser selesai.


In [29]:
MIN_TEST_ARTICLES = {
    "FAO": 10,
    "IITA": 5,
    "WorldFish": 8
}

parser_gate_rows = []
parser_ready_sources = []

for source_name in ready_sources:

    check = (
        parser_test_df[
            parser_test_df["source_website"] == source_name
        ]
        .sort_values("page")
    )

    status = "NOT READY"
    note = ""

    if len(check) != 2:

        note = "Kedua halaman uji belum berhasil diperiksa."

    else:

        count_1 = int(
            check.iloc[0]["article_cards_found"]
        )

        count_2 = int(
            check.iloc[1]["article_cards_found"]
        )

        urls_1 = parser_test_url_sets.get(
            (source_name, 1),
            set()
        )

        urls_2 = parser_test_url_sets.get(
            (source_name, 2),
            set()
        )

        minimum = MIN_TEST_ARTICLES[source_name]

        new_on_page_2 = len(
            urls_2 - urls_1
        )

        minimum_new = max(
            3,
            (len(urls_2) + 1) // 2
        )

        if count_1 < minimum or count_2 < minimum:

            note = (
                f"Kartu artikel kurang dari batas uji "
                f"{minimum} per halaman."
            )

        elif not urls_1 or not urls_2:

            note = "Daftar URL artikel kosong."

        elif new_on_page_2 < minimum_new:

            note = (
                "Halaman kedua terlalu banyak mengulang "
                "artikel dari halaman pertama."
            )

        else:

            status = "READY"
            note = (
                "Jumlah artikel cukup dan URL halaman "
                "pertama serta kedua berbeda."
            )

    if status == "READY":
        parser_ready_sources.append(source_name)

    parser_gate_rows.append({
        "source_website": source_name,
        "status": status,
        "note": note,
    })


parser_gate_df = pd.DataFrame(
    parser_gate_rows
)

display(parser_gate_df)

print(
    "\nSumber yang lolos validasi parser:",
    parser_ready_sources
)

if not parser_ready_sources:
    raise RuntimeError(
        "Tidak ada sumber yang lolos validasi parser. "
        "Jangan jalankan scraping massal."
    )

,source_website,status,note
0,FAO,READY,Jumlah artikel cukup dan URL halaman pertama serta kedua berbeda.
1,IITA,READY,Jumlah artikel cukup dan URL halaman pertama serta kedua berbeda.
2,WorldFish,NOT READY,Halaman kedua terlalu banyak mengulang artikel dari halaman pertama.



Sumber yang lolos validasi parser: ['FAO', 'IITA']


In [30]:
all_records = []
scrape_summary = []

for source_name in parser_ready_sources:

    print(f"\nMulai scraping: {source_name}")

    pages_completed = 0
    article_count = 0
    duplicate_count = 0

    stop_reason = (
        "Batas halaman maksimum tercapai."
    )

    seen_page_urls = set()
    seen_article_urls = set()

    for page_number in range(
        1,
        MAX_PAGES[source_name] + 1
    ):

        current_url = build_page_url(
            source_name,
            page_number
        )

        if current_url in seen_page_urls:

            stop_reason = (
                "URL halaman sudah pernah dikunjungi."
            )

            break

        seen_page_urls.add(current_url)

        try:

            # Selalu periksa robots.txt untuk URL yang diminta.
            if not robots_allowed(
                source_name,
                current_url
            ):

                stop_reason = (
                    f"robots.txt tidak mengizinkan "
                    f"halaman {page_number}."
                )

                break

            html = polite_get(
                source_name,
                current_url
            )

            final_url = request_log[-1]["final_url"]

            page_rows, soup = parse_listing_page(
                html,
                source_name,
                final_url
            )

            if not page_rows:

                stop_reason = (
                    "Parser tidak menemukan artikel "
                    f"pada halaman {page_number}."
                )

                print(
                    f"{source_name}: halaman {page_number} "
                    "tidak menghasilkan artikel."
                )

                break

            # Hanya masukkan URL yang belum pernah ditemukan.
            new_rows = [
                row
                for row in page_rows
                if row["article_url"] not in seen_article_urls
            ]

            duplicate_count += (
                len(page_rows) - len(new_rows)
            )

            # Jika satu halaman tidak menambahkan artikel baru,
            # hentikan supaya tidak mengambil halaman yang sama berulang.
            if page_number > 1 and not new_rows:

                stop_reason = (
                    "Halaman tidak menghasilkan URL artikel baru. "
                    "Kemungkinan pagination mengulang hasil."
                )

                print(
                    f"{source_name}: halaman {page_number} "
                    "tidak menambah artikel unik. Scraping dihentikan."
                )

                break

            # Tambahkan URL baru ke daftar yang sudah ditemukan.
            for row in new_rows:
                seen_article_urls.add(
                    row["article_url"]
                )

            all_records.extend(new_rows)

            pages_completed += 1
            article_count += len(new_rows)

            print(
                f"{source_name}: "
                f"halaman {page_number} | "
                f"kartu ditemukan {len(page_rows)} | "
                f"artikel baru {len(new_rows)} | "
                f"total unik {article_count}"
            )

            # Simpan checkpoint setiap halaman.
            pd.DataFrame(all_records).to_csv(
                "/content/scraping_checkpoint.csv",
                index=False,
                encoding="utf-8-sig"
            )

            if page_number == MAX_PAGES[source_name]:

                stop_reason = (
                    "Batas halaman maksimum tercapai."
                )

        except (
            RobotsUnverifiedError,
            RobotsDeniedError,
            SourceBlockedError
        ) as exc:

            stop_reason = str(exc)

            print(
                f"Scraping {source_name} dihentikan: "
                f"{stop_reason}"
            )

            break

        except Exception as exc:

            stop_reason = (
                f"Error {type(exc).__name__}: {exc}"
            )

            print(
                f"Scraping {source_name} dihentikan: "
                f"{stop_reason}"
            )

            break

    scrape_summary.append({
        "source_website": source_name,
        "pages_completed": pages_completed,
        "article_cards_detected": article_count,
        "duplicate_cards_skipped": duplicate_count,
        "stop_reason": stop_reason,
    })


scrape_summary_df = pd.DataFrame(
    scrape_summary
)

display(scrape_summary_df)

print(
    "\nTotal artikel unik sebelum ETL:",
    len(all_records)
)


Mulai scraping: FAO
FAO: halaman 1 | kartu ditemukan 20 | artikel baru 20 | total unik 20
FAO: halaman 2 | kartu ditemukan 20 | artikel baru 20 | total unik 40
FAO: halaman 3 | kartu ditemukan 20 | artikel baru 20 | total unik 60
FAO: halaman 4 | kartu ditemukan 20 | artikel baru 20 | total unik 80
FAO: halaman 5 | kartu ditemukan 20 | artikel baru 20 | total unik 100
FAO: halaman 6 | kartu ditemukan 20 | artikel baru 20 | total unik 120
FAO: halaman 7 | kartu ditemukan 20 | artikel baru 20 | total unik 140
FAO: halaman 8 | kartu ditemukan 20 | artikel baru 20 | total unik 160
FAO: halaman 9 | kartu ditemukan 20 | artikel baru 20 | total unik 180
FAO: halaman 10 | kartu ditemukan 20 | artikel baru 20 | total unik 200
FAO: halaman 11 | kartu ditemukan 20 | artikel baru 20 | total unik 220
FAO: halaman 12 | kartu ditemukan 20 | artikel baru 20 | total unik 240
FAO: halaman 13 | kartu ditemukan 20 | artikel baru 20 | total unik 260
FAO: halaman 14 | kartu ditemukan 20 | artikel baru 20 |

,source_website,pages_completed,article_cards_detected,duplicate_cards_skipped,stop_reason
0,FAO,50,1000,0,Batas halaman maksimum tercapai.
1,IITA,119,1133,176,Batas halaman maksimum tercapai.



Total artikel unik sebelum ETL: 2133


In [31]:
# Klasifikasi topik berbasis kata kunci. Kategori ini dibuat saat ETL, bukan label resmi situs.

TOPIC_KEYWORDS = {
    "Fisheries & Aquaculture": [
        "fish", "fisheries", "fishery", "aquaculture", "aquatic", "seafood", "blue food",
        "fish farmer", "fishers", "fisher", "tilapia", "shrimp", "seaweed", "hatchery"
    ],
    "Livestock & Animal Health": [
        "livestock", "cattle", "cow", "dairy", "poultry", "goat", "sheep", "pig",
        "animal health", "veterinary", "animal disease", "feed", "ruminant", "meat"
    ],
    "Crop Production": [
        "crop", "crops", "farmer", "farmers", "farming", "agriculture", "agricultural",
        "harvest", "rice", "maize", "corn", "wheat", "cassava", "yam", "banana",
        "plantain", "soybean", "cowpea", "sorghum", "millet", "seed", "soil",
        "fertilizer", "irrigation", "plant health", "pest", "disease", "crop yield",
        "horticulture"
    ],
    "Forestry & Biodiversity": [
        "forest", "forestry", "agroforestry", "biodiversity", "mangrove", "ecosystem",
        "conservation", "deforestation", "land restoration", "wildlife"
    ],
    "Climate & Environment": [
        "climate", "climate change", "climate-resilient", "climate resilient", "drought",
        "flood", "weather", "environment", "resilience", "adaptation", "sustainability",
        "sustainable"
    ],
    "Food Security & Nutrition": [
        "food security", "food system", "food systems", "nutrition", "malnutrition",
        "hunger", "food safety", "diet", "nutritious", "food loss", "food waste"
    ],
    "Markets & Trade": [
        "market", "markets", "trade", "export", "imports", "commodity", "price",
        "value chain", "agribusiness", "agrifood", "supply chain", "investment", "finance"
    ],
    "Agricultural Technology & Research": [
        "research", "technology", "innovation", "digital agriculture", "mechanization",
        "biotechnology", "breeding", "genetic", "improved varieties", "improved seeds",
        "science", "data", "artificial intelligence"
    ],
    "Rural Development & Policy": [
        "rural", "smallholder", "small-scale farmers", "policy", "livelihood",
        "livelihoods", "community", "gender", "youth", "extension", "cooperative",
        "development", "governance"
    ],
}


def classify_topic(title, summary):
    text = f"{title or ''} {summary or ''}".lower()
    matched_categories = []
    all_terms = []

    for category, terms in TOPIC_KEYWORDS.items():
        found = [term for term in terms if term in text]
        if found:
            matched_categories.append(category)
            all_terms.extend(found)

    if not matched_categories:
        return "Other / Institutional", [], False

    # Satu kategori utama per artikel; seluruh kata kunci cocok disimpan di kolom keywords.
    return matched_categories[0], sorted(set(all_terms)), True


if not all_records:
    raise RuntimeError("Belum ada artikel berhasil dikumpulkan. Periksa hasil preflight dan uji parser.")

raw_df = pd.DataFrame(all_records)
raw_df["scraped_at"] = datetime.now(timezone.utc).isoformat()

classified = raw_df.apply(
    lambda row: classify_topic(row.get("title", ""), row.get("summary", "")), axis=1
)
raw_df["topic_category"] = [item[0] for item in classified]
raw_df["keywords"] = ["; ".join(item[1]) for item in classified]
raw_df["is_agriculture_relevant"] = [item[2] for item in classified]

raw_df["published_date"] = pd.to_datetime(raw_df["published_date"], errors="coerce")
raw_df["year"] = raw_df["published_date"].dt.year.astype("Int64")
raw_df["month"] = raw_df["published_date"].dt.month.astype("Int64")
raw_df["quarter"] = raw_df["published_date"].dt.quarter.astype("Int64")


def create_article_id(row):
    key = f"{row['source_website']}|{row['article_url']}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]


raw_df["article_id"] = raw_df.apply(create_article_id, axis=1)

for column in ["title", "summary", "author", "source_category", "article_url",
               "archive_page_url", "keywords"]:
    raw_df[column] = raw_df[column].fillna("").astype(str).map(normalize_space)

clean_df = raw_df[
    (raw_df["title"].str.len() > 5) &
    (raw_df["article_url"].str.startswith("http"))
].copy()

# Deduplikasi URL. Artikel dari dua sumber berbeda tetap dipertahankan jika URL-nya berbeda.
clean_df = clean_df.drop_duplicates(subset=["article_url"], keep="first").reset_index(drop=True)

column_order = [
    "article_id", "source_website", "title", "published_date", "year", "month",
    "quarter", "summary", "author", "source_category", "topic_category",
    "keywords", "is_agriculture_relevant", "article_url", "archive_page_url",
    "scraped_at"
]
clean_df = clean_df[column_order]
tableau_df = clean_df[clean_df["is_agriculture_relevant"] == True].copy()

print("Artikel bersih seluruh kategori:", len(clean_df))
print("Artikel relevan agrikultur:", len(tableau_df))
print("URL unik:", clean_df["article_url"].nunique())
display(clean_df.head(10))


Artikel bersih seluruh kategori: 2133
Artikel relevan agrikultur: 1989
URL unik: 2133


,article_id,source_website,title,published_date,year,month,quarter,summary,author,source_category,topic_category,keywords,is_agriculture_relevant,article_url,archive_page_url,scraped_at
0,34407a26a3464f2f,FAO,FAO urges job creation and coordinated action to tackle hunger and poverty,2026-11-19,2026,11,4,Event at FAO headquarters marks the second anniversary of the Global Alliance against Hunger and Poverty launched at...,,,Food Security & Nutrition,hunger,True,https://www.fao.org/newsroom/detail/fao-urges-job-creation-and-coordinated-action-to-tackle-hunger-and-poverty/en,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
1,8f4e91d052b5efb0,FAO,"FAO Food Outlook: Global food commodity supplies remain generally ample, but risks are mounting",2026-11-19,2026,11,4,"New edition examines fertilizer markets, Black Sea export disruptions, and developments in olive oil, whey, and live...",,,Livestock & Animal Health,commodity; development; export; feed; fertilizer; livestock; market; markets,True,https://www.fao.org/newsroom/detail/fao-food-outlook--global-food-commodity-supplies-remain-generally-ample--but-ris...,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
2,c9e0b6fb6a29566e,FAO,MED-Dialogues 2026: FAO calls for stronger diplomatic action on conflict-driven food crises,2026-11-19,2026,11,4,FAO Director-General highlights three priorities to address food crises in conflict situations,,,Other / Institutional,,False,https://www.fao.org/newsroom/detail/med-dialogues-2026--fao-calls-for-stronger-diplomatic-action-on-conflict-driven-...,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
3,f7f8011997a6c9ed,FAO,Parliamentary Summit against Hunger and Malnutrition ends with call for inclusive implementation,2026-11-18,2026,11,4,Lawmakers from around the world signal commitment to collaboration to ensure the right to food,,,Food Security & Nutrition,hunger; malnutrition; nutrition,True,https://www.fao.org/newsroom/detail/parliamentary-summit-against-hunger-and-malnutrition-ends-with-call-for-inclusiv...,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
4,5bd487437e6fd52f,FAO,Parliamentarians urged to turn hunger commitments into action as global summit opens in Kenya,2026-11-16,2026,11,4,"FAO calls for stronger laws, public investment and accountability to accelerate progress towards ending hunger and m...",,,Food Security & Nutrition,hunger; investment; malnutrition; nutrition,True,https://www.fao.org/newsroom/detail/parliamentarians-urged-to-turn-hunger-commitments-into-action-as-global-summit-o...,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
5,5b81cb5834502eab,FAO,COP31: FAO welcomes endorsement of Antalya Action Plan on Integrated Fire Management,2026-11-16,2026,11,4,Countries turning wildfire commitments into action,,,Other / Institutional,,False,https://www.fao.org/newsroom/detail/cop31-fao-welcomes-endorsement-of-antalya-action-plan-on-integrated-fire-managem...,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
6,207eb40b14ca1e3d,FAO,Conflict and disease outbreaks deepen DR Congo’s hunger crisis as record numbers are projected to face acute food in...,2026-11-13,2026,11,4,"FAO, WFP, UNICEF and WHO call for urgent action as acute hunger is projected to hit more than 30 million, including ...",,,Crop Production,disease; hunger,True,https://www.fao.org/newsroom/detail/conflict-and-disease-outbreaks-deepen-dr-congo-s-hunger-crisis-as-record-numbers...,https://www.fao.org/newsroom/archive/,2026-11-21T19:01:55.556128+00:00
7,9c7e513002ff853b,FAO,UN launches Disaster Risk Financing Platform at COP31 to make humanitarian responses faster and more predictable,2026-11-12,2026,11,4,The initiative aims to secure access to more than $3 billion in risk financing to protect 75 million people from cli...,,,Climate & Environment,climate,True,https://www.fao.org/newsroom/detail/un-launches-disaster-risk-financing-platform

In [32]:
# Pemeriksaan kualitas dan distribusi data.

quality_summary = pd.DataFrame({
    "metric": [
        "Total artikel mentah",
        "Total artikel bersih",
        "Artikel relevan agrikultur",
        "Artikel di luar kategori kata kunci",
        "Artikel tanpa tanggal publikasi",
        "Artikel tanpa ringkasan arsip",
        "URL unik",
        "Target relevan minimum",
    ],
    "value": [
        len(raw_df),
        len(clean_df),
        len(tableau_df),
        int((clean_df["is_agriculture_relevant"] == False).sum()),
        int(clean_df["published_date"].isna().sum()),
        int((clean_df["summary"].str.len() == 0).sum()),
        clean_df["article_url"].nunique(),
        TARGET_MIN_ROWS,
    ]
})
display(quality_summary)

print("\nRingkasan per sumber:")
display(clean_df.groupby("source_website").agg(
    total_articles=("article_id", "count"),
    agriculture_relevant=("is_agriculture_relevant", "sum"),
    date_available=("published_date", "count"),
).reset_index())

print("\nDistribusi kategori utama:")
display(tableau_df["topic_category"].value_counts().rename_axis(
    "topic_category"
).reset_index(name="article_count"))

if len(tableau_df) >= TARGET_MIN_ROWS:
    print(f"TARGET TERCAPAI: {len(tableau_df):,} artikel relevan unik.")
else:
    print(
        f"TARGET BELUM TERCAPAI: {len(tableau_df):,} artikel relevan unik "
        f"dari target {TARGET_MIN_ROWS:,}. Jangan membuat data tambahan; "
        "periksa apakah masih ada halaman yang diizinkan untuk diambil."
    )


,metric,value
0,Total artikel mentah,2133
1,Total artikel bersih,2133
2,Artikel relevan agrikultur,1989
3,Artikel di luar kategori kata kunci,144
4,Artikel tanpa tanggal publikasi,1
5,Artikel tanpa ringkasan arsip,2
6,URL unik,2133
7,Target relevan minimum,1000



Ringkasan per sumber:


,source_website,total_articles,agriculture_relevant,date_available
0,FAO,1000,915,1000
1,IITA,1133,1074,1132



Distribusi kategori utama:


,topic_category,article_count
0,Crop Production,1058
1,Food Security & Nutrition,238
2,Climate & Environment,195
3,Livestock & Animal Health,132
4,Forestry & Biodiversity,120
5,Agricultural Technology & Research,93
6,Fisheries & Aquaculture,60
7,Markets & Trade,55
8,Rural Development & Policy,38


TARGET TERCAPAI: 1,989 artikel relevan unik.


In [33]:
# Ekspor dataset utama, versi khusus Tableau, serta log sumber dan permintaan.

OUTPUT_ALL = "/content/global_agriculture_news_all.csv"
OUTPUT_TABLEAU = "/content/global_agriculture_news_tableau.csv"
OUTPUT_PREFLIGHT = "/content/source_preflight_log.csv"
OUTPUT_SCRAPE_LOG = "/content/scraping_source_log.csv"
OUTPUT_REQUEST_LOG = "/content/scraping_request_log.csv"

clean_df.to_csv(OUTPUT_ALL, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
tableau_df.to_csv(OUTPUT_TABLEAU, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
preflight_df.to_csv(OUTPUT_PREFLIGHT, index=False, encoding="utf-8-sig")
pd.DataFrame(scrape_summary).to_csv(OUTPUT_SCRAPE_LOG, index=False, encoding="utf-8-sig")
pd.DataFrame(request_log).to_csv(OUTPUT_REQUEST_LOG, index=False, encoding="utf-8-sig")

print("File tersimpan:")
for path in [OUTPUT_ALL, OUTPUT_TABLEAU, OUTPUT_PREFLIGHT, OUTPUT_SCRAPE_LOG, OUTPUT_REQUEST_LOG]:
    print(path)

try:
    from google.colab import files
    print("\nMengunduh CSV khusus Tableau...")
    files.download(OUTPUT_TABLEAU)
except Exception:
    print("Jika bukan Google Colab, ambil CSV melalui panel Files pada runtime.")


File tersimpan:
/content/global_agriculture_news_all.csv
/content/global_agriculture_news_tableau.csv
/content/source_preflight_log.csv
/content/scraping_source_log.csv
/content/scraping_request_log.csv

Mengunduh CSV khusus Tableau...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>